# MV Validation — Group-Aware Train / Validation / Test Split

## Objective

This notebook independently reads the permanently saved feature table used by the LightGBM notebook and creates a new `label_split` under one hard constraint:

> All rows belonging to the same `login_id + acct_nbr` pair must stay in exactly one split.

The new split also tries to remain close to the Model Development split in terms of:

1. total row count;
2. positive-label count;
3. positive rate;
4. employee-account pair count.

The notebook does **not** retrain the model and does **not** overwrite any permanent table.

## Source confirmed from the LightGBM notebook

The original LightGBM notebook reads:

```text
insider_us_nms_features_table_v2
```

and defines:

```text
label_split = train / val / test
```

The current split is therefore recovered directly from the saved feature table rather than reconstructed from temporary notebook variables.


In [ ]:
# Core imports

import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import types as T

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Configuration

Only the saved feature table is required.

The weights below define how strongly the allocation algorithm prioritizes each target:

- `ROW_WEIGHT`: closeness to the original row-count allocation;
- `POSITIVE_WEIGHT`: closeness to the original positive-label allocation;
- `PAIR_WEIGHT`: closeness to the original pair-count allocation;
- `OVERSHOOT_WEIGHT`: additional penalty when a split exceeds its target.

Positive balance receives the largest ordinary weight because positive observations are scarce and directly affect model training and recall evaluation.


In [ ]:
# Saved feature table used by the LightGBM notebook
FEATURE_TABLE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdad1s1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_features_table_v2"
)

# Column names
EMPLOYEE_COL = "login_id"
ACCOUNT_COL = "acct_nbr"
TARGET_COL = "label"
ORIGINAL_SPLIT_COL = "label_split"
NEW_SPLIT_COL = "new_label_split"

# Expected split labels from the LightGBM notebook
SPLIT_ORDER = ["train", "val", "test"]

# Reproducibility
RANDOM_SEED = 42

# Objective-function weights
ROW_WEIGHT = 1.0
POSITIVE_WEIGHT = 3.0
PAIR_WEIGHT = 0.25
OVERSHOOT_WEIGHT = 5.0

# Number of randomized greedy attempts.
# More attempts may improve the final allocation but increase runtime.
N_ATTEMPTS = 5

# Small number used to avoid division by zero.
EPSILON = 1e-12

## 2. Read the permanently saved feature table

Only columns required for the split analysis are selected.

No model features are needed because this notebook is analyzing and rebuilding `label_split`, not generating predictions.


In [ ]:
features_df = (
    spark.read
    .format("delta")
    .load(FEATURE_TABLE_PATH)
    .select(
        EMPLOYEE_COL,
        ACCOUNT_COL,
        TARGET_COL,
        ORIGINAL_SPLIT_COL
    )
)

# Standardize the target to integer 0/1 for aggregation.
features_df = features_df.withColumn(
    TARGET_COL,
    F.col(TARGET_COL).cast("int")
)

display(features_df.limit(10))

print("Total rows:", features_df.count())

## 3. Recover the original MD split distribution

This table establishes the target allocation.

For each original split, the notebook calculates:

- number of rows;
- row proportion;
- number of positive rows;
- share of all positive rows;
- number of negative rows;
- positive rate;
- number of distinct employee-account pairs.

These original proportions become the targets for the new group-aware split.


In [ ]:
total_rows = features_df.count()
total_positives = (
    features_df
    .agg(F.sum(TARGET_COL).alias("total_positives"))
    .first()["total_positives"]
)
total_negatives = total_rows - total_positives

original_split_summary = (
    features_df
    .groupBy(ORIGINAL_SPLIT_COL)
    .agg(
        F.count("*").alias("row_count"),
        F.sum(TARGET_COL).alias("positive_count"),
        (
            F.count("*") - F.sum(TARGET_COL)
        ).alias("negative_count"),
        F.avg(TARGET_COL).alias("positive_rate"),
        F.countDistinct(
            F.struct(EMPLOYEE_COL, ACCOUNT_COL)
        ).alias("pair_count")
    )
    .withColumn(
        "row_share",
        F.col("row_count") / F.lit(total_rows)
    )
    .withColumn(
        "positive_share",
        F.col("positive_count") / F.lit(total_positives)
    )
    .orderBy(
        F.when(F.col(ORIGINAL_SPLIT_COL) == "train", 1)
         .when(F.col(ORIGINAL_SPLIT_COL) == "val", 2)
         .when(F.col(ORIGINAL_SPLIT_COL) == "test", 3)
         .otherwise(4)
    )
)

display(original_split_summary)

## 4. Check the current employee-account leakage

A pair is considered cross-split when the same `login_id + acct_nbr` appears in more than one of:

```text
train / val / test
```

This check quantifies the current issue before creating a replacement split.


In [ ]:
pair_current_split_check = (
    features_df
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.count("*").alias("row_count"),
        F.sum(TARGET_COL).alias("positive_count"),
        F.countDistinct(
            ORIGINAL_SPLIT_COL
        ).alias("number_of_splits"),
        F.collect_set(
            ORIGINAL_SPLIT_COL
        ).alias("splits")
    )
)

current_leakage_summary = (
    pair_current_split_check
    .agg(
        F.count("*").alias("total_pairs"),
        F.sum(
            F.when(
                F.col("number_of_splits") > 1,
                1
            ).otherwise(0)
        ).alias("pairs_crossing_splits"),
        F.sum(
            F.when(
                F.col("number_of_splits") > 1,
                F.col("row_count")
            ).otherwise(0)
        ).alias("rows_in_crossing_pairs"),
        F.sum(
            F.when(
                F.col("number_of_splits") > 1,
                F.col("positive_count")
            ).otherwise(0)
        ).alias("positives_in_crossing_pairs")
    )
    .withColumn(
        "crossing_pair_rate",
        F.col("pairs_crossing_splits") / F.col("total_pairs")
    )
    .withColumn(
        "crossing_row_rate",
        F.col("rows_in_crossing_pairs") / F.lit(total_rows)
    )
    .withColumn(
        "crossing_positive_rate",
        F.col("positives_in_crossing_pairs") / F.lit(total_positives)
    )
)

display(current_leakage_summary)

## 5. Create one record per employee-account pair

Each pair becomes one indivisible allocation unit.

For every pair, the notebook calculates:

- `row_count`;
- `positive_count`;
- `negative_count`;
- `pair_count = 1`.

Once a pair is assigned to a split, every original row from that pair receives the same new split.


In [ ]:
pair_summary_spk = (
    features_df
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.count("*").alias("row_count"),
        F.sum(TARGET_COL).alias("positive_count")
    )
    .withColumn(
        "negative_count",
        F.col("row_count") - F.col("positive_count")
    )
)

print("Number of employee-account pairs:", pair_summary_spk.count())

# The allocation loop is easier to express in pandas.
# Approximately several hundred thousand pair-level rows are usually manageable.
pair_pd = pair_summary_spk.toPandas()

pair_pd["row_count"] = pair_pd["row_count"].astype(np.int64)
pair_pd["positive_count"] = pair_pd["positive_count"].astype(np.int64)
pair_pd["negative_count"] = pair_pd["negative_count"].astype(np.int64)

display(pair_pd.head(10))

## 6. Build the allocation targets from the original split

The new split should remain as close as possible to the original MD split.

For each split, targets are based on the original:

- share of total rows;
- share of all positive rows;
- share of distinct employee-account pairs.

Using the original shares is preferable to manually assuming 70/15/15 because it reproduces the actual saved MD allocation.


In [ ]:
original_target_pd = (
    original_split_summary
    .select(
        F.col(ORIGINAL_SPLIT_COL).alias("split"),
        "row_count",
        "positive_count",
        "pair_count",
        "row_share",
        "positive_share"
    )
    .toPandas()
)

# Keep only train / val / test and preserve their intended order.
original_target_pd = (
    original_target_pd[
        original_target_pd["split"].isin(SPLIT_ORDER)
    ]
    .set_index("split")
    .loc[SPLIT_ORDER]
    .reset_index()
)

total_pairs = len(pair_pd)

# Pair targets are based on the original pair-count shares.
original_pair_total = original_target_pd["pair_count"].sum()
original_target_pd["pair_share"] = (
    original_target_pd["pair_count"] / original_pair_total
)

targets = {}

for _, row in original_target_pd.iterrows():
    split = row["split"]

    targets[split] = {
        "rows": float(row["row_share"] * total_rows),
        "positives": float(row["positive_share"] * total_positives),
        "pairs": float(row["pair_share"] * total_pairs)
    }

display(original_target_pd)

print("Allocation targets:")
for split in SPLIT_ORDER:
    print(split, targets[split])

## 7. Objective function

For each candidate split, the algorithm estimates the result after adding the current pair.

The candidate score is:

```text
score =
    ROW_WEIGHT      × normalized row error
  + POSITIVE_WEIGHT × normalized positive-count error
  + PAIR_WEIGHT     × normalized pair-count error
  + OVERSHOOT_WEIGHT × normalized overshoot penalty
```

Where:

```text
normalized row error
= |projected rows - target rows| / target rows
```

The same form is used for positive count and pair count.

The overshoot penalty discourages continuing to add pairs to a split that has already exceeded its target.

The pair is assigned to the split with the lowest score.

This is a greedy approximation, not a mathematical proof of the global optimum. Multiple randomized attempts are used, and the allocation with the lowest final objective is retained.


In [ ]:
def normalized_absolute_error(actual, target):
    """Return absolute deviation as a proportion of the target."""

    return abs(actual - target) / max(target, EPSILON)


def normalized_overshoot(actual, target):
    """Penalize only the amount above the target."""

    return max(actual - target, 0.0) / max(target, EPSILON)


def candidate_score(
    projected_rows,
    projected_positives,
    projected_pairs,
    target_values
):
    """Calculate the allocation penalty for one candidate split."""

    row_error = normalized_absolute_error(
        projected_rows,
        target_values["rows"]
    )

    positive_error = normalized_absolute_error(
        projected_positives,
        target_values["positives"]
    )

    pair_error = normalized_absolute_error(
        projected_pairs,
        target_values["pairs"]
    )

    overshoot_penalty = (
        normalized_overshoot(
            projected_rows,
            target_values["rows"]
        )
        + normalized_overshoot(
            projected_positives,
            target_values["positives"]
        )
        + normalized_overshoot(
            projected_pairs,
            target_values["pairs"]
        )
    )

    return (
        ROW_WEIGHT * row_error
        + POSITIVE_WEIGHT * positive_error
        + PAIR_WEIGHT * pair_error
        + OVERSHOOT_WEIGHT * overshoot_penalty
    )


def final_allocation_objective(current_totals, targets):
    """Calculate one overall objective after all pairs are assigned."""

    total_score = 0.0

    for split in SPLIT_ORDER:
        total_score += candidate_score(
            projected_rows=current_totals[split]["rows"],
            projected_positives=current_totals[split]["positives"],
            projected_pairs=current_totals[split]["pairs"],
            target_values=targets[split]
        )

    return total_score

## 8. Greedy group-aware allocation

Important implementation details:

1. Pair integrity is guaranteed because allocation occurs once per pair.
2. Positive-heavy and large pairs are assigned first because they are hardest to balance.
3. Random noise is used only to break ties between similarly sized pairs.
4. Several attempts are run, and the allocation with the best final objective is kept.


In [ ]:
def allocate_pairs_greedily(
    pair_data,
    targets,
    random_seed
):
    """Assign each employee-account pair to exactly one split."""

    work = pair_data.copy()

    rng = np.random.default_rng(random_seed)

    # Random tie-breaker prevents the original dataframe order from
    # determining the allocation of otherwise similar pairs.
    work["_tie_breaker"] = rng.random(len(work))

    # Allocate the most difficult pairs first:
    # first by positive count, then row count, then random tie-breaker.
    work = work.sort_values(
        ["positive_count", "row_count", "_tie_breaker"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    current = {
        split: {
            "rows": 0.0,
            "positives": 0.0,
            "pairs": 0.0
        }
        for split in SPLIT_ORDER
    }

    assignments = []

    for row in work.itertuples(index=False):

        best_split = None
        best_score = np.inf

        for split in SPLIT_ORDER:

            projected_rows = (
                current[split]["rows"]
                + row.row_count
            )

            projected_positives = (
                current[split]["positives"]
                + row.positive_count
            )

            projected_pairs = (
                current[split]["pairs"]
                + 1
            )

            score = candidate_score(
                projected_rows=projected_rows,
                projected_positives=projected_positives,
                projected_pairs=projected_pairs,
                target_values=targets[split]
            )

            if score < best_score:
                best_score = score
                best_split = split

        assignments.append(best_split)

        current[best_split]["rows"] += row.row_count
        current[best_split]["positives"] += row.positive_count
        current[best_split]["pairs"] += 1

    work[NEW_SPLIT_COL] = assignments

    objective = final_allocation_objective(
        current_totals=current,
        targets=targets
    )

    return (
        work.drop(columns=["_tie_breaker"]),
        current,
        objective
    )


best_pair_assignment = None
best_current_totals = None
best_objective = np.inf
attempt_results = []

for attempt in range(N_ATTEMPTS):

    attempt_seed = RANDOM_SEED + attempt

    assignment, current_totals, objective = (
        allocate_pairs_greedily(
            pair_data=pair_pd,
            targets=targets,
            random_seed=attempt_seed
        )
    )

    attempt_results.append({
        "attempt": attempt + 1,
        "random_seed": attempt_seed,
        "objective": objective
    })

    if objective < best_objective:
        best_pair_assignment = assignment
        best_current_totals = current_totals
        best_objective = objective

attempt_results_pd = pd.DataFrame(attempt_results)

display(attempt_results_pd)

print("Best objective:", best_objective)
display(best_pair_assignment.head(10))

## 9. Join the new split back to every original row

The assignment table has one row per pair.

After joining it back, every original observation receives the split assigned to its employee-account pair.


In [ ]:
assignment_schema = T.StructType([
    T.StructField(EMPLOYEE_COL, T.StringType(), True),
    T.StructField(ACCOUNT_COL, T.StringType(), True),
    T.StructField(NEW_SPLIT_COL, T.StringType(), False)
])

# Cast keys to string before Spark conversion to avoid pandas type ambiguity.
assignment_for_spark = best_pair_assignment[
    [EMPLOYEE_COL, ACCOUNT_COL, NEW_SPLIT_COL]
].copy()

assignment_for_spark[EMPLOYEE_COL] = (
    assignment_for_spark[EMPLOYEE_COL]
    .astype("string")
)

assignment_for_spark[ACCOUNT_COL] = (
    assignment_for_spark[ACCOUNT_COL]
    .astype("string")
)

pair_assignment_spk = spark.createDataFrame(
    assignment_for_spark,
    schema=assignment_schema
)

features_with_new_split = (
    features_df
    .withColumn(
        EMPLOYEE_COL,
        F.col(EMPLOYEE_COL).cast("string")
    )
    .withColumn(
        ACCOUNT_COL,
        F.col(ACCOUNT_COL).cast("string")
    )
    .join(
        pair_assignment_spk,
        on=[EMPLOYEE_COL, ACCOUNT_COL],
        how="left"
    )
)

display(features_with_new_split.limit(10))

## 10. Hard-constraint validation

The following checks must pass:

1. no pair is missing a new split;
2. each pair appears in exactly one new split;
3. no employee-account pair crosses train / val / test.

A feasible result must have:

```text
missing assignments = 0
pairs crossing new splits = 0
```


In [ ]:
missing_assignment_count = (
    features_with_new_split
    .filter(F.col(NEW_SPLIT_COL).isNull())
    .count()
)

new_pair_integrity = (
    features_with_new_split
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.countDistinct(
            NEW_SPLIT_COL
        ).alias("number_of_new_splits")
    )
)

pairs_crossing_new_splits = (
    new_pair_integrity
    .filter(F.col("number_of_new_splits") > 1)
    .count()
)

hard_constraint_summary = pd.DataFrame([{
    "missing_assignments": missing_assignment_count,
    "pairs_crossing_new_splits": pairs_crossing_new_splits,
    "hard_constraint_passed": (
        missing_assignment_count == 0
        and pairs_crossing_new_splits == 0
    )
}])

display(hard_constraint_summary)

## 11. Compare the original and new split distributions

This is the main feasibility output.

The new allocation is feasible when:

1. pair integrity is fully satisfied;
2. row shares remain close to the original;
3. positive shares remain close to the original;
4. positive rates remain close to the original;
5. validation and test retain enough positive observations for stable evaluation.


In [ ]:
new_split_summary = (
    features_with_new_split
    .groupBy(NEW_SPLIT_COL)
    .agg(
        F.count("*").alias("new_row_count"),
        F.sum(TARGET_COL).alias("new_positive_count"),
        (
            F.count("*") - F.sum(TARGET_COL)
        ).alias("new_negative_count"),
        F.avg(TARGET_COL).alias("new_positive_rate"),
        F.countDistinct(
            F.struct(EMPLOYEE_COL, ACCOUNT_COL)
        ).alias("new_pair_count")
    )
    .withColumn(
        "new_row_share",
        F.col("new_row_count") / F.lit(total_rows)
    )
    .withColumn(
        "new_positive_share",
        F.col("new_positive_count") / F.lit(total_positives)
    )
)

comparison_summary = (
    original_split_summary
    .select(
        F.col(ORIGINAL_SPLIT_COL).alias("split"),
        F.col("row_count").alias("original_row_count"),
        F.col("row_share").alias("original_row_share"),
        F.col("positive_count").alias("original_positive_count"),
        F.col("positive_share").alias("original_positive_share"),
        F.col("positive_rate").alias("original_positive_rate"),
        F.col("pair_count").alias("original_pair_count")
    )
    .join(
        new_split_summary.select(
            F.col(NEW_SPLIT_COL).alias("split"),
            "new_row_count",
            "new_row_share",
            "new_positive_count",
            "new_positive_share",
            "new_positive_rate",
            "new_pair_count"
        ),
        on="split",
        how="inner"
    )
    .withColumn(
        "row_share_difference",
        F.col("new_row_share")
        - F.col("original_row_share")
    )
    .withColumn(
        "positive_share_difference",
        F.col("new_positive_share")
        - F.col("original_positive_share")
    )
    .withColumn(
        "positive_rate_difference",
        F.col("new_positive_rate")
        - F.col("original_positive_rate")
    )
    .orderBy(
        F.when(F.col("split") == "train", 1)
         .when(F.col("split") == "val", 2)
         .when(F.col("split") == "test", 3)
         .otherwise(4)
    )
)

display(comparison_summary)

## 12. Simple feasibility criteria

These thresholds are validation guidelines rather than universal statistical laws.

Suggested criteria:

- pair integrity: exactly zero crossing pairs;
- absolute row-share difference: no more than 2 percentage points;
- absolute positive-share difference: no more than 2 percentage points;
- positive-rate relative difference: no more than 10%;
- validation and test must each contain at least a configurable minimum number of positives.

The minimum-positive threshold should be selected according to the project's evaluation requirements.


In [ ]:
# Configurable review thresholds
MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE = 0.02
MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE = 0.02
MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE = 0.10
MIN_POSITIVES_IN_VAL_OR_TEST = 100

comparison_pd = comparison_summary.toPandas()

comparison_pd["absolute_row_share_difference"] = (
    comparison_pd["row_share_difference"].abs()
)

comparison_pd["absolute_positive_share_difference"] = (
    comparison_pd["positive_share_difference"].abs()
)

comparison_pd["relative_positive_rate_difference"] = (
    (
        comparison_pd["new_positive_rate"]
        - comparison_pd["original_positive_rate"]
    ).abs()
    / comparison_pd["original_positive_rate"].replace(0, np.nan)
)

comparison_pd["row_balance_passed"] = (
    comparison_pd["absolute_row_share_difference"]
    <= MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE
)

comparison_pd["positive_share_balance_passed"] = (
    comparison_pd["absolute_positive_share_difference"]
    <= MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE
)

comparison_pd["positive_rate_balance_passed"] = (
    comparison_pd["relative_positive_rate_difference"]
    <= MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE
)

comparison_pd["minimum_positive_count_passed"] = True

comparison_pd.loc[
    comparison_pd["split"].isin(["val", "test"]),
    "minimum_positive_count_passed"
] = (
    comparison_pd.loc[
        comparison_pd["split"].isin(["val", "test"]),
        "new_positive_count"
    ]
    >= MIN_POSITIVES_IN_VAL_OR_TEST
)

all_balance_checks_passed = bool(
    comparison_pd[
        [
            "row_balance_passed",
            "positive_share_balance_passed",
            "positive_rate_balance_passed",
            "minimum_positive_count_passed"
        ]
    ]
    .all()
    .all()
)

hard_constraint_passed = bool(
    hard_constraint_summary.loc[
        0,
        "hard_constraint_passed"
    ]
)

overall_feasible = (
    hard_constraint_passed
    and all_balance_checks_passed
)

display(comparison_pd)

print("Hard constraint passed:", hard_constraint_passed)
print("All balance checks passed:", all_balance_checks_passed)
print("Overall split considered feasible:", overall_feasible)

## 13. Optional diagnostic: largest pairs

Large employee-account pairs can make perfect balancing impossible because they cannot be divided.

This table identifies pairs that have the greatest influence on row and positive-label allocation.


In [ ]:
largest_pairs = (
    pair_summary_spk
    .orderBy(
        F.desc("positive_count"),
        F.desc("row_count")
    )
)

display(largest_pairs.limit(50))

## 14. Optional save step

The notebook intentionally does not overwrite the original feature table.

After review and approval, the new pair assignment may be saved separately.

The code below is commented out to prevent accidental permanent writes.


In [ ]:
# OPTIONAL_OUTPUT_PATH = (
#     "abfss://.../ins_us_nms/v1/output/"
#     "mv_employee_account_group_split_assignment_v1"
# )
#
# (
#     pair_assignment_spk
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .save(OPTIONAL_OUTPUT_PATH)
# )
#
# print("Saved pair assignment to:", OPTIONAL_OUTPUT_PATH)

## Interpretation

The result supports a **Yes** answer to issue A when:

1. every employee-account pair is assigned to exactly one split;
2. the new row allocation remains close to the original MD allocation;
3. positive counts and positive rates remain acceptably balanced;
4. validation and test retain enough positive observations.

After feasibility is established, the next validation step is to retrain the LightGBM model using `new_label_split` and compare its in-time and OOT performance with the original MD model.
